In [6]:
from google.colab import files
files.upload()  # upload kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"prateekshetty28","key":"253cf17c53d9ab0465b6311ee47cdce7"}'}

In [7]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [8]:
!kaggle datasets download -d mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot
!unzip edgeiiotset-cyber-security-dataset-of-iot-iiot.zip

Dataset URL: https://www.kaggle.com/datasets/mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot
License(s): CC-BY-NC-SA-4.0
100% 1.63G/1.63G [00:14<00:00, 117MB/s]

Archive:  edgeiiotset-cyber-security-dataset-of-iot-iiot.zip
  inflating: Edge-IIoTset dataset/Attack traffic/Backdoor_attack.csv  
  inflating: Edge-IIoTset dataset/Attack traffic/Backdoor_attack.pcap  
  inflating: Edge-IIoTset dataset/Attack traffic/DDoS HTTP Flood Attacks.pcap  
  inflating: Edge-IIoTset dataset/Attack traffic/DDoS ICMP Flood Attacks.pcap  
  inflating: Edge-IIoTset dataset/Attack traffic/DDoS TCP SYN Flood Attacks.pcap  
  inflating: Edge-IIoTset dataset/Attack traffic/DDoS UDP Flood Attacks.pcap  
  inflating: Edge-IIoTset dataset/Attack traffic/DDoS_HTTP_Flood_attack.csv  
  inflating: Edge-IIoTset dataset/Attack traffic/DDoS_ICMP_Flood_attack.csv  
  inflating: Edge-IIoTset dataset/Attack traffic/DDoS_TCP_SYN_Flood_attack.csv  
  inflating: Edge-IIoTset dataset/Attack traffic/DDoS_UDP

In [9]:
import pandas as pd
import numpy as np

# LOAD
df = pd.read_csv(
    "Edge-IIoTset dataset/Selected dataset for ML and DL/ML-EdgeIIoT-dataset.csv",
    nrows=50000,
    low_memory=False
)

print("Initial Shape:", df.shape)

# SHUFFLE
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# ==============================
# USE MULTI-CLASS (IMPORTANT)
# ==============================
y = df["Attack_type"]

# DROP LABELS
df = df.drop(["Attack_label", "Attack_type"], axis=1)

# KEEP NUMERIC
df = df.select_dtypes(include=[np.number])

# CLEAN
df = df.replace([np.inf, -np.inf], np.nan)
df = df.fillna(0)

X = df

# REMOVE LOW VARIANCE
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(0.001)
X = selector.fit_transform(X)

# SCALE
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

# SPLIT
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# MODEL
from lightgbm import LGBMClassifier
model = LGBMClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1
)

model.fit(X_train, y_train)

# EVAL
from sklearn.metrics import accuracy_score
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"✅ FINAL ACCURACY: {acc*100:.2f}%")

Initial Shape: (50000, 63)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005795 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1791
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 24
[LightGBM] [Info] Start training from score -1.554831
[LightGBM] [Info] Start training from score -2.168273
[LightGBM] [Info] Start training from score -3.910774
[LightGBM] [Info] Start training from score -3.718308
[LightGBM] [Info] Start training from score -1.520969
[LightGBM] [Info] Start training from score -1.578787
[LightGBM] [Info] Start training from score -1.582918
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warni

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ FINAL ACCURACY: 91.57%


In [10]:
import joblib

joblib.dump(model, "centralized_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Saved successfully")

Saved successfully


In [11]:
import os
print(os.listdir())

['.config', 'scaler.pkl', 'Edge_IIoTset__DatasetFL.pdf', 'Edge-IIoTset dataset', 'Readme.txt', 'edgeiiotset-cyber-security-dataset-of-iot-iiot.zip', 'centralized_model.pkl', 'sample_data']


In [12]:
from google.colab import files

files.download("centralized_model.pkl")
files.download("scaler.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>